# INV-M365-F: Failure-Closed / No Side Effects

**Invariant proven:** INV-M365-F-001

**Lemma referenced:** LEM-M365-F-001-01

**Phase 1:** docs/math/instruction_contract.md (Section 6)

## 1. Setup

When Eval(instruction, tenant_state) is undefined, tenant state remains unchanged (no observable mutation).

In [ ]:
P = {"Admin", "User"}
A_Admin, A_User = {"admin_mutate", "admin_read"}, {"user_read"}


def is_valid(instruction):
    if not isinstance(instruction, (tuple, list)) or len(instruction) != 4:
        return False
    p, a = instruction[0], instruction[2]
    return (p == "Admin" and a in A_Admin) or (p == "User" and a in A_User)


def eval_spec(instruction, tenant_state):
    if not is_valid(instruction):
        return None
    p, a = instruction[0], instruction[2]
    if a in {"admin_read", "user_read"}:
        return ("Success", tenant_state, [])
    if a == "admin_mutate" and p == "Admin":
        return ("Success", tenant_state + 1, [{}])
    return None


tau_before = 100
x_invalid = ("User", "i1", "admin_mutate", {})

## 2. Lemma Execution (LEM-M365-F-001-01)

When Eval is undefined, no state change: tenant state used by any subsequent evaluation is unchanged from tenant_state.

In [ ]:
out = eval_spec(x_invalid, tau_before)
assert out is None
tau_after = tau_before
assert tau_after == tau_before, "Lemma: no state change on rejection"

## 3. Invariant Verification

**INV-M365-F-001 pass_condition:** Whenever Eval(instruction, tenant_state) is undefined, the tenant state used by any subsequent evaluation is unchanged from tenant_state.

In [ ]:
def verify_F001(instruction, tenant_state, eval_fn):
    out = eval_fn(instruction, tenant_state)
    if out is None:
        assert True
        return tenant_state
    return out[1]


state_after_reject = verify_F001(x_invalid, tau_before, eval_spec)
assert state_after_reject == tau_before
state_after_second = verify_F001(x_invalid, state_after_reject, eval_spec)
assert state_after_second == tau_before
print("INV-M365-F-001: pass_condition holds.")

## 4. Failure Demonstration

Invalid instruction (wrong structure, or User+admin_mutate): Eval undefined -> observable state must remain tau.

In [ ]:
for bad in [(1, 2, 3), ("User", "i1", "admin_mutate", {}), ("Admin", "i1", "user_read", {})]:
    o = eval_spec(bad, tau_before)
    if o is None:
        assert tau_before == 100
print("Failure cases: no state change on rejection.")

## 5. Conclusion

INV-M365-F-001 holds: invalid or rejected instructions cause no tenant state change.

In [ ]:
print("VERIFY: INV-M365-F-001 — pass.")